In [121]:
from google.oauth2.service_account import Credentials
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
import pandas as pd

from tabulate import tabulate
import pathlib
import shutil

import pygetwindow as gw
import pydirectinput as di
import pyautogui as ag
import time
import keyboard as kb

In [122]:
BASE_PATH            = pathlib.Path.cwd().resolve().parent.parent                  # ! <--- Will Change
DESKTOP_PATH         = pathlib.Path.home() / "Desktop"                             # * default export location
LOG_PATH             = DESKTOP_PATH / "Logs"                                       # ! <--- Will Change
ERROR_PATH           = BASE_PATH / "project" / "outputs" / "errors.txt"            #
SERVICE_ACCOUNT_PATH = BASE_PATH / "project" / "keys" / "service-account-key.json" #
RECORDING_PATH       = BASE_PATH / "project" / "outputs" / "recordings"            #

In [ ]:
while True:
    print(di.position())

(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(2147, 113)
(214

In [ ]:
UI_POSITIONS = {

    # Display
    "Display": (49,130),
    "Minimum": (184,333),
    "Center":  (310,333),
    "Maximum": (448,333),

    # Load Data
    "Load Data": (49,226),
    "Browse":    (490,100),
    "Path":      (890,72),
    "Select":    (1060,672),

    # Advanced
    "Advanced":    (49,333),
    "Start frame": (280,170),
    "End frame":   (280,210),
    "Output FPS":  (280,250),
    "Export":      (180,105),

    # Preview
    "Frame #": (750,50),
    "Speed":   (1295,55),
    "2x":      (1290,160),
    "Play":    (1353,55),

}

def setup() -> None:
    window = gw.getWindowsWithTitle("CTS Playback v1.3.vi")[0]
    window.moveTo(0,0)

setup()

In [ ]:
UI_CHECKS = [

    # Display
    ((142,166), True),  # Upsample
    ((142,210), False), # Mirror
    ((405,255), False), # Dynamic
    ((137,640), True),  # High Contrast Mode
    ((340,640), True),  # Spatial Filter
    ((325,685), False), # Hotspot Detection

    # Preview
    ((1170,55),  False), # Loop
    ((1066,622), False), # Min
    ((1166,622), False), # Max
    ((1266,622), False), # Cursor

]

OFF_COLORS:list[tuple[int,int,int]] = [(45,45,45),(51,51,51)]

def check_UI():

    di.click(*UI_POSITIONS["Display"])

    for position, expected in UI_CHECKS:
        pixel:tuple[int,int,int] = ag.screenshot(region=(*position,1,1)).getpixel([0,0])
        current:bool = False if pixel in OFF_COLORS else True
        if current != expected:
            di.click(*position, interval=0.5)

    di.click(*UI_POSITIONS["Advanced"])

    di.doubleClick(*UI_POSITIONS["Output FPS"])

    di.write("4")

check_UI()

In [ ]:
class Entry:

    def __init__(self,
        id:int,
        # type:str, # ! Auto Later
        number:int,
        start:int,
        first:int,
        last:int,
        min:float,
        center:float,
        max:float
    ):
        self.id = id
        self.type = "Fall" # ! Auto Later
        self.number = number
        self.start = start
        self.first = first
        self.last = last
        self.min = min
        self.center = center
        self.max = max

    def __repr__(self) -> str:
        return f"{self.id}-{self.type}-{self.number}"

    def to_str(self) -> str:
        return f"{self} | {self.start}/{self.first}/{self.last} | {self.min}/{self.center}/{self.max}"

In [ ]:
def relocate(entry:Entry) -> None:
    file = list(DESKTOP_PATH.glob("raw*.avi"))[0]
    name = f"{entry}.avi"
    path = RECORDING_PATH / str(entry.id)
    folder = path.mkdir(exist_ok=True) # * don't yell if it's there
    shutil.move(str(file), path / name)

def record(entry:Entry) -> None:

    di.click(UI_POSITIONS["Display"])

    # Temp stuff

    di.click(*UI_POSITIONS["Advanced"])

    # ! if red then click
    di.click(*UI_POSITIONS["Play"])

    di.doubleClick(*UI_POSITIONS["Frame #"])
    di.write(str(entry.start - 1))

    di.doubleClick(*UI_POSITIONS["Start frame"])
    di.write(str(entry.start - 1))

    di.doubleClick(*UI_POSITIONS["End frame"])
    di.write(str(entry.last - 1))

    di.click(*UI_POSITIONS["Export"])

    time.sleep(30) # await for message

    relocate(entry)

def register(entries:list[Entry]):

    di.click(*UI_POSITIONS["Load Data"])
    time.sleep(0.5)
    di.click(*UI_POSITIONS["Browse"])

    window = gw.getWindowsWithTitle("Choose or Enter Path of Folder")[0]
    window.moveTo(0,0)

    di.click(*UI_POSITIONS["Select"])

In [ ]:
SHEET_ID = "1xxY6bhoceU7rLoHfdBupSE_5sZAFO5w--ygXLBXTr8c" # ! <--- Will Change
PAGE_NAME = "PRE"                                         # ! <--- Will Change

credentials = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_PATH,
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)

client = gspread.authorize(credentials)

sheet = client.open_by_key(SHEET_ID)

page = sheet.worksheet(PAGE_NAME)

data = get_as_dataframe(page)

data = data.dropna(subset=["ID", "Fall #", "Start", "First", "Last"])

data["Min"]    = data["Min"].fillna(0)
data["Center"] = data["Center"].fillna(22.5)
data["Max"]    = data["Max"].fillna(27.5)

In [ ]:
entries:list[list[Entry]] = []
current:int = int(data["ID"][0])
set:list[Entry] = []

for _, row in data.iterrows():

    id = int(row["ID"])

    if id != current:
        current = id
        entries.append(set)
        set = []

    # TODO: type
    number = int(row["Fall #"])
    start = int(row["Start"])
    first = int(row["First"])
    last = int(row["Last"])

    if start > first or first > last:
        print("ERROR")
        continue

    minimum = row["Min"]
    center  = row["Center"]
    maximum = row["Max"]

    if minimum > center or center > maximum:
        print("ERROR")
        continue

    if minimum != 0:
        print("ERROR")
        continue

    set.append(Entry(id, number, start, first, last, minimum, center, maximum))

# for set in entries:
#     for entry in set:
#         print(entry.to_str())
#     print("= = = = =")

In [ ]:
for entry in entries[0]:
    record(entry)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process